# Notebook 03 — Training & Evaluation

Train every model (Popularity, MF-BPR, LightGCN, SASRec, SASGCN, **AdaFuse**), evaluate with full-item ranking, report mean ± std over 5 seeds, and run paired significance tests.

**Prerequisite:** Run `01_data_preprocessing.ipynb` first — this notebook loads the processed data it saves.

---
## 1. Setup & Load Data

In [ ]:
import os, sys, time, random, pickle, math, copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.sparse as sp
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Paths
PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "roadmap" / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
CKPT_DIR = RESULTS_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "tables").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "logs").mkdir(parents=True, exist_ok=True)

In [ ]:
# Load processed data produced by notebook 01
PROCESSED_PATH = DATA_DIR / "processed" / "amazon_beauty_processed.pkl"

with open(PROCESSED_PATH, "rb") as f:
    data = pickle.load(f)

num_users = data["num_users"]
num_items = data["num_items"]
train_dict = data["train_dict"]      # {user_id: [item_id, ...]}
val_dict   = data["val_dict"]        # {user_id: item_id}
test_dict  = data["test_dict"]       # {user_id: item_id}
train_sequences = data["train_sequences"]  # {user_id: [item_id, ...]} time-ordered

print(f"Users: {num_users}  |  Items: {num_items}")
print(f"Train interactions: {sum(len(v) for v in train_dict.values())}")
print(f"Val users: {len(val_dict)}  |  Test users: {len(test_dict)}")

In [ ]:
# Build sparse adjacency for graph-based models
def build_adj_matrix(train_dict, num_users, num_items):
    """Build symmetric normalized adjacency for LightGCN."""
    rows, cols = [], []
    for u, items in train_dict.items():
        for i in items:
            rows.append(u)
            cols.append(num_users + i)  # offset item ids
    rows, cols = np.array(rows), np.array(cols)
    # Bidirectional
    all_rows = np.concatenate([rows, cols])
    all_cols = np.concatenate([cols, rows])
    n = num_users + num_items
    adj = sp.coo_matrix(
        (np.ones(len(all_rows)), (all_rows, all_cols)),
        shape=(n, n),
    ).tocsr()

    # Symmetric normalization D^{-1/2} A D^{-1/2}
    deg = np.array(adj.sum(axis=1)).flatten()
    deg_inv_sqrt = np.where(deg > 0, np.power(deg, -0.5), 0.0)
    D_inv_sqrt = sp.diags(deg_inv_sqrt)
    norm_adj = D_inv_sqrt @ adj @ D_inv_sqrt
    norm_adj = norm_adj.tocoo()

    indices = torch.LongTensor(np.stack([norm_adj.row, norm_adj.col]))
    values = torch.FloatTensor(norm_adj.data)
    sparse_adj = torch.sparse_coo_tensor(indices, values, torch.Size([n, n]))
    return sparse_adj.to(DEVICE)

norm_adj = build_adj_matrix(train_dict, num_users, num_items)
print(f"Adjacency: {norm_adj.shape}, nnz={norm_adj._nnz()}")

---
## 2. Evaluation Metrics

In [ ]:
def recall_at_k(ranked_items, ground_truth, k):
    """Recall@K for a single user. ground_truth is a set."""
    top_k = set(ranked_items[:k])
    return len(top_k & ground_truth) / max(len(ground_truth), 1)


def ndcg_at_k(ranked_items, ground_truth, k):
    """NDCG@K for a single user."""
    top_k = ranked_items[:k]
    dcg = sum(
        1.0 / math.log2(rank + 2)
        for rank, item in enumerate(top_k)
        if item in ground_truth
    )
    idcg = sum(
        1.0 / math.log2(rank + 2)
        for rank in range(min(len(ground_truth), k))
    )
    return dcg / idcg if idcg > 0 else 0.0


def hit_rate_at_k(ranked_items, ground_truth, k):
    """HR@K: 1 if any ground-truth item appears in top K."""
    return 1.0 if set(ranked_items[:k]) & ground_truth else 0.0


@torch.no_grad()
def evaluate_model(score_fn, test_data, train_dict, num_items,
                   k_values=(10, 20), batch_size=256):
    """
    Full-item ranking evaluation.

    Parameters
    ----------
    score_fn : callable(user_ids: LongTensor) -> FloatTensor[B, num_items]
        Returns scores for all items per user.
    test_data : dict  {user_id: item_id} or {user_id: set(item_ids)}
    train_dict : dict {user_id: list[item_id]}
    num_items : int

    Returns
    -------
    metrics : dict   {"Recall@10": float, ...}
    per_user : dict  {"NDCG@10": np.array, ...}  (for significance tests)
    """
    users = sorted(test_data.keys())
    per_user = {f"{m}@{k}": [] for k in k_values for m in ("Recall", "NDCG", "HR")}

    for start in range(0, len(users), batch_size):
        batch_users = users[start : start + batch_size]
        user_tensor = torch.LongTensor(batch_users).to(DEVICE)
        scores = score_fn(user_tensor)  # [B, num_items]

        # Mask training items
        for idx, u in enumerate(batch_users):
            train_items = train_dict.get(u, [])
            if len(train_items) > 0:
                scores[idx, train_items] = -1e9

        for k in k_values:
            _, topk_indices = torch.topk(scores, k, dim=1)
            topk_indices = topk_indices.cpu().numpy()

            for idx, u in enumerate(batch_users):
                gt = test_data[u]
                gt_set = gt if isinstance(gt, set) else {gt}
                ranked = topk_indices[idx].tolist()

                per_user[f"Recall@{k}"].append(recall_at_k(ranked, gt_set, k))
                per_user[f"NDCG@{k}"].append(ndcg_at_k(ranked, gt_set, k))
                per_user[f"HR@{k}"].append(hit_rate_at_k(ranked, gt_set, k))

    per_user = {key: np.array(vals) for key, vals in per_user.items()}
    metrics = {key: vals.mean() for key, vals in per_user.items()}
    return metrics, per_user

---
## 3. Training Utilities

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
class BPRDataset(Dataset):
    """Sample (user, pos_item, neg_item) triples for BPR training."""

    def __init__(self, train_dict, num_items):
        self.train_dict = train_dict
        self.num_items = num_items
        self.users = []
        self.pos_items = []
        for u, items in train_dict.items():
            for i in items:
                self.users.append(u)
                self.pos_items.append(i)
        self.user_set = {u: set(items) for u, items in train_dict.items()}

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        pos = self.pos_items[idx]
        neg = random.randint(0, self.num_items - 1)
        while neg in self.user_set[u]:
            neg = random.randint(0, self.num_items - 1)
        return u, pos, neg

In [ ]:
class SequenceDataset(Dataset):
    """Produce fixed-length sequences for SASRec-style models."""

    def __init__(self, train_sequences, num_items, max_len=50):
        self.seqs = train_sequences
        self.num_items = num_items
        self.max_len = max_len
        self.users = list(train_sequences.keys())

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        seq = self.seqs[u]
        # input = seq[:-1], target = seq[-1]
        input_seq = seq[:-1]
        target = seq[-1]
        # Truncate / pad to max_len
        if len(input_seq) > self.max_len:
            input_seq = input_seq[-self.max_len :]
        pad_len = self.max_len - len(input_seq)
        # 0 = padding token; item ids are 0-based, so shift by +1 inside model
        padded = [0] * pad_len + [i + 1 for i in input_seq]  # +1 so 0 is pad
        return u, torch.LongTensor(padded), target

In [ ]:
class EarlyStopping:
    def __init__(self, patience=50, metric="NDCG@10", mode="max"):
        self.patience = patience
        self.metric = metric
        self.mode = mode
        self.best_score = -float("inf") if mode == "max" else float("inf")
        self.counter = 0
        self.best_state = None
        self.stopped = False

    def step(self, score, model):
        improved = (score > self.best_score) if self.mode == "max" else (score < self.best_score)
        if improved:
            self.best_score = score
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
        return improved

    def load_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

In [ ]:
def bpr_loss(pos_scores, neg_scores):
    return -F.logsigmoid(pos_scores - neg_scores).mean()


def train_bpr_epoch(model, dataloader, optimizer, weight_decay=1e-4):
    model.train()
    total_loss = 0.0
    for users, pos, neg in dataloader:
        users = users.to(DEVICE)
        pos = pos.to(DEVICE)
        neg = neg.to(DEVICE)

        pos_scores, neg_scores = model.bpr_forward(users, pos, neg)
        loss = bpr_loss(pos_scores, neg_scores)

        # L2 regularisation on embeddings
        reg = model.l2_reg(users, pos, neg) * weight_decay
        loss = loss + reg

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * users.size(0)

    return total_loss / len(dataloader.dataset)

In [ ]:
def full_training_loop(
    model, train_loader, optimizer, weight_decay,
    val_data, train_dict, num_items,
    epochs=200, patience=50, eval_every=5,
    model_name="model",
):
    """
    Generic BPR training loop with early stopping.

    Returns
    -------
    history : dict  {"loss": [...], "val_ndcg10": [...]}
    """
    es = EarlyStopping(patience=patience, metric="NDCG@10")
    history = {"loss": [], "val_ndcg10": []}

    pbar = tqdm(range(1, epochs + 1), desc=model_name)
    for epoch in pbar:
        loss = train_bpr_epoch(model, train_loader, optimizer, weight_decay)
        history["loss"].append(loss)

        if epoch % eval_every == 0:
            model.eval()
            metrics, _ = evaluate_model(
                model.score_all_items, val_data, train_dict, num_items,
                k_values=(10,),
            )
            val_ndcg = metrics["NDCG@10"]
            history["val_ndcg10"].append(val_ndcg)
            improved = es.step(val_ndcg, model)
            pbar.set_postfix(loss=f"{loss:.4f}", val_ndcg10=f"{val_ndcg:.4f}",
                             best=f"{es.best_score:.4f}", patience=es.counter)
            if es.stopped:
                print(f"Early stopping at epoch {epoch}")
                break

    es.load_best(model)
    return history

---
## 4. Model Definitions

All models expose a consistent interface:
- `bpr_forward(users, pos_items, neg_items)` → `(pos_scores, neg_scores)`
- `l2_reg(users, pos_items, neg_items)` → scalar
- `score_all_items(user_ids)` → `[B, num_items]`

### 4a. Popularity Baseline

In [ ]:
class PopularityModel:
    """Non-personalised baseline: rank by item frequency."""

    def __init__(self, train_dict, num_items):
        counts = np.zeros(num_items, dtype=np.float32)
        for items in train_dict.values():
            for i in items:
                counts[i] += 1
        self.scores = torch.from_numpy(counts).to(DEVICE)

    def score_all_items(self, user_ids):
        return self.scores.unsqueeze(0).expand(user_ids.size(0), -1)

### 4b. MF-BPR

In [ ]:
class MFBPR(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        nn.init.xavier_normal_(self.user_emb.weight)
        nn.init.xavier_normal_(self.item_emb.weight)

    def bpr_forward(self, users, pos, neg):
        u = self.user_emb(users)
        p = self.item_emb(pos)
        n = self.item_emb(neg)
        return (u * p).sum(dim=1), (u * n).sum(dim=1)

    def l2_reg(self, users, pos, neg):
        u = self.user_emb(users)
        p = self.item_emb(pos)
        n = self.item_emb(neg)
        return (u.norm(2).pow(2) + p.norm(2).pow(2) + n.norm(2).pow(2)) / users.size(0)

    @torch.no_grad()
    def score_all_items(self, user_ids):
        u = self.user_emb(user_ids)
        return u @ self.item_emb.weight.T

### 4c. LightGCN

In [ ]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64, n_layers=3, adj=None):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.n_layers = n_layers
        self.embedding = nn.Embedding(num_users + num_items, embed_dim)
        nn.init.xavier_normal_(self.embedding.weight)
        self.adj = adj  # sparse normalized adjacency on device

    def forward(self):
        """Multi-layer propagation, return mean of all layers."""
        e = self.embedding.weight
        layer_embs = [e]
        for _ in range(self.n_layers):
            e = torch.sparse.mm(self.adj, e)
            layer_embs.append(e)
        final = torch.stack(layer_embs, dim=0).mean(dim=0)
        user_emb = final[: self.num_users]
        item_emb = final[self.num_users :]
        return user_emb, item_emb

    def bpr_forward(self, users, pos, neg):
        user_emb, item_emb = self.forward()
        u = user_emb[users]
        p = item_emb[pos]
        n = item_emb[neg]
        return (u * p).sum(dim=1), (u * n).sum(dim=1)

    def l2_reg(self, users, pos, neg):
        u = self.embedding.weight[users]
        p = self.embedding.weight[self.num_users + pos]
        n = self.embedding.weight[self.num_users + neg]
        return (u.norm(2).pow(2) + p.norm(2).pow(2) + n.norm(2).pow(2)) / users.size(0)

    @torch.no_grad()
    def score_all_items(self, user_ids):
        user_emb, item_emb = self.forward()
        u = user_emb[user_ids]
        return u @ item_emb.T

### 4d. SASRec

In [ ]:
class SASRec(nn.Module):
    def __init__(self, num_items, embed_dim=64, max_len=50,
                 n_layers=2, n_heads=1, dropout=0.2):
        super().__init__()
        self.num_items = num_items
        self.max_len = max_len
        # +1 for padding token at index 0
        self.item_emb = nn.Embedding(num_items + 1, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, dropout=dropout,
            activation="gelu", batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def encode_sequence(self, seq):
        """seq: [B, max_len] with 0=pad, 1..num_items = item ids."""
        B, L = seq.shape
        positions = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(seq) + self.pos_emb(positions)
        x = self.emb_norm(self.dropout(x))

        # Causal mask
        causal_mask = nn.Transformer.generate_square_subsequent_mask(L, device=seq.device)
        # Padding mask
        pad_mask = (seq == 0)  # True where padded

        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        # Take the last non-padded position
        # For simplicity use the last position (sequences are right-aligned)
        return x[:, -1, :]  # [B, embed_dim]

    def bpr_forward(self, users, pos, neg, seqs=None):
        """seqs must be provided externally; users is ignored."""
        u = self.encode_sequence(seqs)
        # Items are 0-based in pos/neg, but item_emb has +1 offset
        p = self.item_emb(pos + 1)
        n = self.item_emb(neg + 1)
        return (u * p).sum(dim=1), (u * n).sum(dim=1)

    def l2_reg(self, users, pos, neg, **kwargs):
        p = self.item_emb(pos + 1)
        n = self.item_emb(neg + 1)
        return (p.norm(2).pow(2) + n.norm(2).pow(2)) / pos.size(0)

    @torch.no_grad()
    def score_all_items_with_seq(self, seqs):
        u = self.encode_sequence(seqs)
        all_items = self.item_emb.weight[1:]  # skip pad token
        return u @ all_items.T

In [ ]:
# SASRec needs its own training loop because it uses sequences, not (u,i) pairs.

def train_sasrec_epoch(model, dataloader, optimizer, num_items, weight_decay=1e-4):
    model.train()
    total_loss = 0.0
    for users, seqs, targets in dataloader:
        seqs = seqs.to(DEVICE)
        targets = targets.to(DEVICE)
        B = seqs.size(0)

        # Sample negatives
        neg = torch.randint(0, num_items, (B,), device=DEVICE)

        pos_scores, neg_scores = model.bpr_forward(None, targets, neg, seqs=seqs)
        loss = bpr_loss(pos_scores, neg_scores)
        reg = model.l2_reg(None, targets, neg) * weight_decay
        loss = loss + reg

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * B

    return total_loss / len(dataloader.dataset)


def build_seq_for_users(user_ids, train_sequences, max_len=50):
    """Build padded sequences for a batch of users (for evaluation)."""
    seqs = []
    for u in user_ids.cpu().tolist():
        s = train_sequences.get(u, [])
        if len(s) > max_len:
            s = s[-max_len:]
        pad_len = max_len - len(s)
        padded = [0] * pad_len + [i + 1 for i in s]
        seqs.append(padded)
    return torch.LongTensor(seqs).to(DEVICE)


def full_training_loop_sasrec(
    model, train_loader, optimizer, weight_decay,
    val_data, train_dict, train_sequences, num_items,
    max_len=50, epochs=200, patience=50, eval_every=5,
    model_name="SASRec",
):
    es = EarlyStopping(patience=patience)
    history = {"loss": [], "val_ndcg10": []}

    def score_fn(user_ids):
        seqs = build_seq_for_users(user_ids, train_sequences, max_len)
        return model.score_all_items_with_seq(seqs)

    pbar = tqdm(range(1, epochs + 1), desc=model_name)
    for epoch in pbar:
        loss = train_sasrec_epoch(model, train_loader, optimizer, num_items, weight_decay)
        history["loss"].append(loss)

        if epoch % eval_every == 0:
            model.eval()
            metrics, _ = evaluate_model(score_fn, val_data, train_dict, num_items, k_values=(10,))
            val_ndcg = metrics["NDCG@10"]
            history["val_ndcg10"].append(val_ndcg)
            improved = es.step(val_ndcg, model)
            pbar.set_postfix(loss=f"{loss:.4f}", val_ndcg10=f"{val_ndcg:.4f}",
                             best=f"{es.best_score:.4f}", patience=es.counter)
            if es.stopped:
                print(f"Early stopping at epoch {epoch}")
                break

    es.load_best(model)
    return history

### 4e. SASGCN (Baseline Hybrid: GCN + SASRec + Concat Fusion)

In [ ]:
class VanillaGCN(nn.Module):
    """Standard GCN with weight matrices and ReLU (as used in SASGCN)."""

    def __init__(self, num_users, num_items, embed_dim=64, n_layers=2, adj=None):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.n_layers = n_layers
        self.embedding = nn.Embedding(num_users + num_items, embed_dim)
        nn.init.xavier_normal_(self.embedding.weight)
        self.weights = nn.ModuleList([
            nn.Linear(embed_dim, embed_dim) for _ in range(n_layers)
        ])
        self.adj = adj

    def forward(self):
        e = self.embedding.weight
        for W in self.weights:
            e = torch.sparse.mm(self.adj, e)
            e = F.relu(W(e))
        user_emb = e[: self.num_users]
        item_emb = e[self.num_users :]
        return user_emb, item_emb


class SASGCN(nn.Module):
    """SASGCN: GCN graph + SASRec sequential, concat+MLP fusion."""

    def __init__(self, num_users, num_items, embed_dim=64,
                 gcn_layers=2, sasrec_layers=2, sasrec_heads=1,
                 max_len=50, dropout=0.2, adj=None):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.max_len = max_len

        self.gcn = VanillaGCN(num_users, num_items, embed_dim, gcn_layers, adj)
        self.sasrec = SASRec(num_items, embed_dim, max_len, sasrec_layers, sasrec_heads, dropout)

        # Concat fusion: 2*embed_dim -> embed_dim
        self.fusion_mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def fuse(self, e_seq, e_col):
        return self.fusion_mlp(torch.cat([e_seq, e_col], dim=-1))

    def bpr_forward(self, users, pos, neg, seqs=None):
        user_emb_graph, item_emb_graph = self.gcn.forward()
        e_col = user_emb_graph[users]
        e_seq = self.sasrec.encode_sequence(seqs)
        e_fused = self.fuse(e_seq, e_col)

        p = item_emb_graph[pos]
        n = item_emb_graph[neg]
        return (e_fused * p).sum(dim=1), (e_fused * n).sum(dim=1)

    def l2_reg(self, users, pos, neg, **kwargs):
        u = self.gcn.embedding.weight[users]
        p = self.gcn.embedding.weight[self.num_users + pos]
        n = self.gcn.embedding.weight[self.num_users + neg]
        return (u.norm(2).pow(2) + p.norm(2).pow(2) + n.norm(2).pow(2)) / users.size(0)

    @torch.no_grad()
    def score_all_items_with_seq(self, user_ids, seqs):
        user_emb_graph, item_emb_graph = self.gcn.forward()
        e_col = user_emb_graph[user_ids]
        e_seq = self.sasrec.encode_sequence(seqs)
        e_fused = self.fuse(e_seq, e_col)
        return e_fused @ item_emb_graph.T

### 4f. AdaFuse (Ours: LightGCN + SASRec + Attention Gate)

In [ ]:
class AdaFuse(nn.Module):
    """
    Adaptive Attention Fusion of Sequential and Collaborative Representations.

    Combines LightGCN (graph) + SASRec (sequence) via a learned per-user
    attention gate with residual formulation:
        alpha(u) = sigmoid(w^T [e_seq; e_col; e_seq * e_col])
        e_final  = e_col + alpha * (e_seq - e_col)
    alpha=0 recovers pure graph; regularising alpha toward 0 defaults to graph.
    """

    def __init__(self, num_users, num_items, embed_dim=64,
                 lgcn_layers=3, sasrec_layers=2, sasrec_heads=1,
                 max_len=50, dropout=0.2, adj=None):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embed_dim = embed_dim
        self.max_len = max_len

        self.lightgcn = LightGCN(num_users, num_items, embed_dim, lgcn_layers, adj)
        self.sasrec = SASRec(num_items, embed_dim, max_len, sasrec_layers, sasrec_heads, dropout)

        # Attention gate: maps 3*embed_dim -> 1
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 3, 1),
        )

    def fuse(self, e_seq, e_col):
        gate_input = torch.cat([e_seq, e_col, e_seq * e_col], dim=-1)
        alpha = torch.sigmoid(self.gate(gate_input))  # [B, 1]
        # Residual: graph is the base, sequence corrects it. alpha=0 → pure graph.
        e_fused = e_col + alpha * (e_seq - e_col)
        return e_fused, alpha

    def bpr_forward(self, users, pos, neg, seqs=None):
        user_emb_graph, item_emb_graph = self.lightgcn.forward()
        e_col = user_emb_graph[users]
        e_seq = self.sasrec.encode_sequence(seqs)
        e_fused, _ = self.fuse(e_seq, e_col)

        # Score items using LightGCN's item embeddings
        p = item_emb_graph[pos]
        n = item_emb_graph[neg]
        return (e_fused * p).sum(dim=1), (e_fused * n).sum(dim=1)

    def l2_reg(self, users, pos, neg, **kwargs):
        u = self.lightgcn.embedding.weight[users]
        p = self.lightgcn.embedding.weight[self.num_users + pos]
        n = self.lightgcn.embedding.weight[self.num_users + neg]
        return (u.norm(2).pow(2) + p.norm(2).pow(2) + n.norm(2).pow(2)) / users.size(0)

    @torch.no_grad()
    def score_all_items_with_seq(self, user_ids, seqs):
        user_emb_graph, item_emb_graph = self.lightgcn.forward()
        e_col = user_emb_graph[user_ids]
        e_seq = self.sasrec.encode_sequence(seqs)
        e_fused, _ = self.fuse(e_seq, e_col)
        return e_fused @ item_emb_graph.T

    @torch.no_grad()
    def get_alphas(self, user_ids, seqs):
        """Return fusion weights for analysis."""
        user_emb_graph, _ = self.lightgcn.forward()
        e_col = user_emb_graph[user_ids]
        e_seq = self.sasrec.encode_sequence(seqs)
        _, alpha = self.fuse(e_seq, e_col)
        return alpha.squeeze(-1)

In [ ]:
# Shared training loop for hybrid models (SASGCN, AdaFuse)
# They need both BPR triples AND sequences.

class HybridDataset(Dataset):
    """Produces (user, pos_item, neg_item, padded_sequence) tuples."""

    def __init__(self, train_dict, train_sequences, num_items, max_len=50):
        self.num_items = num_items
        self.max_len = max_len
        self.user_set = {u: set(items) for u, items in train_dict.items()}
        self.users = []
        self.pos_items = []
        self.sequences = {}
        for u, items in train_dict.items():
            seq = train_sequences.get(u, items)
            if len(seq) > max_len:
                seq = seq[-max_len:]
            pad_len = max_len - len(seq)
            self.sequences[u] = [0] * pad_len + [i + 1 for i in seq]
            for i in items:
                self.users.append(u)
                self.pos_items.append(i)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        pos = self.pos_items[idx]
        neg = random.randint(0, self.num_items - 1)
        while neg in self.user_set[u]:
            neg = random.randint(0, self.num_items - 1)
        return u, pos, neg, torch.LongTensor(self.sequences[u])


def train_hybrid_epoch(model, dataloader, optimizer, weight_decay=1e-4):
    model.train()
    total_loss = 0.0
    for users, pos, neg, seqs in dataloader:
        users = users.to(DEVICE)
        pos = pos.to(DEVICE)
        neg = neg.to(DEVICE)
        seqs = seqs.to(DEVICE)

        pos_scores, neg_scores = model.bpr_forward(users, pos, neg, seqs=seqs)
        loss = bpr_loss(pos_scores, neg_scores)
        reg = model.l2_reg(users, pos, neg) * weight_decay
        loss = loss + reg

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * users.size(0)

    return total_loss / len(dataloader.dataset)


def full_training_loop_hybrid(
    model, train_loader, optimizer, weight_decay,
    val_data, train_dict, train_sequences, num_items,
    max_len=50, epochs=200, patience=50, eval_every=5,
    model_name="Hybrid",
):
    es = EarlyStopping(patience=patience)
    history = {"loss": [], "val_ndcg10": []}

    def score_fn(user_ids):
        seqs = build_seq_for_users(user_ids, train_sequences, max_len)
        return model.score_all_items_with_seq(user_ids, seqs)

    pbar = tqdm(range(1, epochs + 1), desc=model_name)
    for epoch in pbar:
        loss = train_hybrid_epoch(model, train_loader, optimizer, weight_decay)
        history["loss"].append(loss)

        if epoch % eval_every == 0:
            model.eval()
            metrics, _ = evaluate_model(score_fn, val_data, train_dict, num_items, k_values=(10,))
            val_ndcg = metrics["NDCG@10"]
            history["val_ndcg10"].append(val_ndcg)
            improved = es.step(val_ndcg, model)
            pbar.set_postfix(loss=f"{loss:.4f}", val_ndcg10=f"{val_ndcg:.4f}",
                             best=f"{es.best_score:.4f}", patience=es.counter)
            if es.stopped:
                print(f"Early stopping at epoch {epoch}")
                break

    es.load_best(model)
    return history

---
## 5. Multi-Seed Evaluation Framework

In [ ]:
SEEDS = [42, 123, 456, 789, 1024]
K_VALUES = (10, 20)
EMBED_DIM = 64
MAX_LEN = 50
EPOCHS = 200
PATIENCE = 50
EVAL_EVERY = 5

In [ ]:
def run_multi_seed_bpr(model_class, model_kwargs, lr, weight_decay, batch_size,
                       seeds=SEEDS, model_name="model"):
    """
    Train a BPR-based model (MF-BPR or LightGCN) over multiple seeds.

    Returns
    -------
    results_df : DataFrame with mean/std per metric
    all_per_user : list[dict]  per-user scores from each seed (for t-tests)
    all_histories : list[dict]
    """
    all_metrics = []
    all_per_user = []
    all_histories = []

    for seed in seeds:
        print(f"\n{'='*50}")
        print(f"{model_name}  |  seed={seed}")
        print(f"{'='*50}")
        set_seed(seed)

        model = model_class(**model_kwargs).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        dataset = BPRDataset(train_dict, num_items)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            num_workers=0, drop_last=False)

        history = full_training_loop(
            model, loader, optimizer, weight_decay,
            val_dict, train_dict, num_items,
            epochs=EPOCHS, patience=PATIENCE, eval_every=EVAL_EVERY,
            model_name=f"{model_name} [seed={seed}]",
        )
        all_histories.append(history)

        model.eval()
        metrics, per_user = evaluate_model(
            model.score_all_items, test_dict, train_dict, num_items,
            k_values=K_VALUES,
        )
        all_metrics.append(metrics)
        all_per_user.append(per_user)
        print(f"  Test: {metrics}")

    df = pd.DataFrame(all_metrics)
    summary = pd.DataFrame({
        "mean": df.mean(),
        "std": df.std(),
    })
    summary["display"] = summary.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
    return summary, all_per_user, all_histories

In [ ]:
def run_multi_seed_seq(model_class, model_kwargs, lr, weight_decay, batch_size,
                       seeds=SEEDS, model_name="SASRec"):
    """Multi-seed for pure sequential models (SASRec)."""
    all_metrics = []
    all_per_user = []
    all_histories = []

    for seed in seeds:
        print(f"\n{'='*50}")
        print(f"{model_name}  |  seed={seed}")
        print(f"{'='*50}")
        set_seed(seed)

        model = model_class(**model_kwargs).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        dataset = SequenceDataset(train_sequences, num_items, MAX_LEN)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            num_workers=0, drop_last=False)

        history = full_training_loop_sasrec(
            model, loader, optimizer, weight_decay,
            val_dict, train_dict, train_sequences, num_items,
            max_len=MAX_LEN, epochs=EPOCHS, patience=PATIENCE, eval_every=EVAL_EVERY,
            model_name=f"{model_name} [seed={seed}]",
        )
        all_histories.append(history)

        model.eval()
        def score_fn(user_ids):
            seqs = build_seq_for_users(user_ids, train_sequences, MAX_LEN)
            return model.score_all_items_with_seq(seqs)

        metrics, per_user = evaluate_model(
            score_fn, test_dict, train_dict, num_items,
            k_values=K_VALUES,
        )
        all_metrics.append(metrics)
        all_per_user.append(per_user)
        print(f"  Test: {metrics}")

    df = pd.DataFrame(all_metrics)
    summary = pd.DataFrame({"mean": df.mean(), "std": df.std()})
    summary["display"] = summary.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
    return summary, all_per_user, all_histories

In [ ]:
def run_multi_seed_hybrid(model_class, model_kwargs, lr, weight_decay, batch_size,
                          seeds=SEEDS, model_name="Hybrid"):
    """Multi-seed for hybrid models (SASGCN, AdaFuse)."""
    all_metrics = []
    all_per_user = []
    all_histories = []

    for seed in seeds:
        print(f"\n{'='*50}")
        print(f"{model_name}  |  seed={seed}")
        print(f"{'='*50}")
        set_seed(seed)

        model = model_class(**model_kwargs).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        dataset = HybridDataset(train_dict, train_sequences, num_items, MAX_LEN)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            num_workers=0, drop_last=False)

        history = full_training_loop_hybrid(
            model, loader, optimizer, weight_decay,
            val_dict, train_dict, train_sequences, num_items,
            max_len=MAX_LEN, epochs=EPOCHS, patience=PATIENCE, eval_every=EVAL_EVERY,
            model_name=f"{model_name} [seed={seed}]",
        )
        all_histories.append(history)

        model.eval()
        def score_fn(user_ids):
            seqs = build_seq_for_users(user_ids, train_sequences, MAX_LEN)
            return model.score_all_items_with_seq(user_ids, seqs)

        metrics, per_user = evaluate_model(
            score_fn, test_dict, train_dict, num_items,
            k_values=K_VALUES,
        )
        all_metrics.append(metrics)
        all_per_user.append(per_user)
        print(f"  Test: {metrics}")

    df = pd.DataFrame(all_metrics)
    summary = pd.DataFrame({"mean": df.mean(), "std": df.std()})
    summary["display"] = summary.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
    return summary, all_per_user, all_histories

---
## 6. Run All Models

Each subsection trains one model across 5 seeds.

### 6a. Popularity

In [ ]:
pop_model = PopularityModel(train_dict, num_items)
pop_metrics, pop_per_user = evaluate_model(
    pop_model.score_all_items, test_dict, train_dict, num_items, k_values=K_VALUES,
)
print("Popularity:", pop_metrics)

pop_summary = pd.DataFrame({
    "mean": pop_metrics,
    "std": {k: 0.0 for k in pop_metrics},
})
pop_summary["display"] = pop_summary["mean"].apply(lambda x: f"{x:.4f}")

### 6b. MF-BPR

In [ ]:
mf_summary, mf_per_user, mf_histories = run_multi_seed_bpr(
    model_class=MFBPR,
    model_kwargs={"num_users": num_users, "num_items": num_items, "embed_dim": EMBED_DIM},
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=1024,
    model_name="MF-BPR",
)
print(mf_summary["display"])

### 6c. LightGCN

In [ ]:
lgcn_summary, lgcn_per_user, lgcn_histories = run_multi_seed_bpr(
    model_class=LightGCN,
    model_kwargs={
        "num_users": num_users, "num_items": num_items,
        "embed_dim": EMBED_DIM, "n_layers": 3, "adj": norm_adj,
    },
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=1024,
    model_name="LightGCN",
)
print(lgcn_summary["display"])

### 6d. SASRec

In [ ]:
sasrec_summary, sasrec_per_user, sasrec_histories = run_multi_seed_seq(
    model_class=SASRec,
    model_kwargs={
        "num_items": num_items, "embed_dim": EMBED_DIM,
        "max_len": MAX_LEN, "n_layers": 2, "n_heads": 1, "dropout": 0.2,
    },
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    model_name="SASRec",
)
print(sasrec_summary["display"])

### 6e. SASGCN

In [ ]:
sasgcn_summary, sasgcn_per_user, sasgcn_histories = run_multi_seed_hybrid(
    model_class=SASGCN,
    model_kwargs={
        "num_users": num_users, "num_items": num_items,
        "embed_dim": EMBED_DIM, "gcn_layers": 2, "sasrec_layers": 2,
        "sasrec_heads": 1, "max_len": MAX_LEN, "dropout": 0.2, "adj": norm_adj,
    },
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    model_name="SASGCN",
)
print(sasgcn_summary["display"])

### 6f. AdaFuse (Ours)

In [ ]:
adafuse_summary, adafuse_per_user, adafuse_histories = run_multi_seed_hybrid(
    model_class=AdaFuse,
    model_kwargs={
        "num_users": num_users, "num_items": num_items,
        "embed_dim": EMBED_DIM, "lgcn_layers": 3, "sasrec_layers": 2,
        "sasrec_heads": 1, "max_len": MAX_LEN, "dropout": 0.2, "adj": norm_adj,
    },
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    model_name="AdaFuse",
)
print(adafuse_summary["display"])

---
## 7. Main Results Table

In [ ]:
all_summaries = {
    "Popularity": pop_summary,
    "MF-BPR": mf_summary,
    "LightGCN": lgcn_summary,
    "SASRec": sasrec_summary,
    "SASGCN": sasgcn_summary,
    "AdaFuse": adafuse_summary,
}

# Build display table
rows = []
for name, summary in all_summaries.items():
    row = {"Model": name}
    for metric in summary.index:
        row[metric] = summary.loc[metric, "display"]
    rows.append(row)

results_table = pd.DataFrame(rows).set_index("Model")
print("\n" + "=" * 80)
print("MAIN RESULTS TABLE")
print("=" * 80)
print(results_table.to_string())

results_table.to_csv(RESULTS_DIR / "tables" / "main_results.csv")
print(f"\nSaved to {RESULTS_DIR / 'tables' / 'main_results.csv'}")

---
## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_histories = {
    "MF-BPR": mf_histories,
    "LightGCN": lgcn_histories,
    "SASRec": sasrec_histories,
    "SASGCN": sasgcn_histories,
    "AdaFuse": adafuse_histories,
}

colors = {
    "MF-BPR": "#94a3b8",
    "LightGCN": "#06b6d4",
    "SASRec": "#a78bfa",
    "SASGCN": "#f59e0b",
    "AdaFuse": "#10b981",
}

# Loss curves (seed 0 only for clarity)
ax = axes[0]
for name, histories in model_histories.items():
    ax.plot(histories[0]["loss"], label=name, color=colors[name], alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("BPR Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(True, alpha=0.2)

# Validation NDCG@10 curves
ax = axes[1]
for name, histories in model_histories.items():
    vals = histories[0]["val_ndcg10"]
    x = [(i + 1) * EVAL_EVERY for i in range(len(vals))]
    ax.plot(x, vals, label=name, color=colors[name], alpha=0.8, marker="o", markersize=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val NDCG@10")
ax.set_title("Validation NDCG@10")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {RESULTS_DIR / 'figures' / 'training_curves.png'}")

---
## 9. Statistical Significance Tests

In [ ]:
from scipy import stats

def paired_ttest(per_user_a, per_user_b, metric="NDCG@10"):
    """
    Paired t-test between two models using per-user scores from seed 0.

    Returns (t_stat, p_value, significant).
    """
    a = per_user_a[0][metric]  # seed 0
    b = per_user_b[0][metric]
    t_stat, p_val = stats.ttest_rel(a, b)
    return t_stat, p_val, p_val < 0.05


print("Paired t-test: AdaFuse vs each baseline (NDCG@10, seed 0)")
print("-" * 60)

baselines_per_user = {
    "MF-BPR": mf_per_user,
    "LightGCN": lgcn_per_user,
    "SASRec": sasrec_per_user,
    "SASGCN": sasgcn_per_user,
}

sig_rows = []
for name, bl_pu in baselines_per_user.items():
    t_stat, p_val, sig = paired_ttest(adafuse_per_user, bl_pu, "NDCG@10")
    sig_rows.append({
        "Baseline": name,
        "t-statistic": f"{t_stat:.3f}",
        "p-value": f"{p_val:.6f}",
        "Significant (p<0.05)": "Yes" if sig else "No",
    })
    print(f"  AdaFuse vs {name:12s} | t={t_stat:+.3f}  p={p_val:.6f}  {'*' if sig else ''}")

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(RESULTS_DIR / "tables" / "significance_tests.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'tables' / 'significance_tests.csv'}")

---
## 10. Summary

All models trained and evaluated. Next steps:

- **Notebook 04**: Ablation studies (w/o GNN, w/o Transformer, w/o attention gate, w/o element-wise product)
- **Notebook 05**: Sparsity-stratified evaluation, attention-weight analysis, embedding visualisation